# Weather forecasts from OpenWeatherMap API

API documentation: https://openweathermap.org/api/forecast5?collection=current_forecast 

## Libraries and Constants

In [5]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [6]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
weather_key = os.getenv("WEATHER_KEY")

In [7]:
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.52,13.41
1,2,Hamburg,Germany,53.55,10.00
2,3,Munich,Germany,48.14,11.58


## One city, one forecast

Start small by calling the API on a single location and exploring the returned data for one time

In [8]:
# select one row of cities_df
berlin = cities_df.loc[0]

url = "https://api.openweathermap.org/data/2.5/forecast"
params = {"lat": berlin["latitude"], "lon" :berlin["longitude"], "appid":weather_key}

response = requests.get(url, params=params)
response

<Response [200]>

A response code of `200` means a successful request! Let's start exploring.

In [9]:
# transform output, check keys
response_json = response.json()
response_json.keys()

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

Let's go through the keys one at time to understand the data structure

In [10]:
# looks like the status code
response_json['cod']

'200'

In [11]:
# ???
response_json['message']

0

In [12]:
# one forecast every 3 hours means 8 per day; five days times 8 makes 40 forecasts
response_json['cnt']

40

In [13]:
# looks like the forecasts themselves
response_json['list']

[{'dt': 1792422000,
  'main': {'temp': 284.75,
   'feels_like': 283.56,
   'temp_min': 284,
   'temp_max': 284.75,
   'pressure': 1031,
   'sea_level': 1031,
   'grnd_level': 1025,
   'humidity': 61,
   'temp_kf': 0.75,
   'dew_point': 277.49},
  'weather': [{'id': 801,
    'main': 'Clouds',
    'description': 'few clouds',
    'icon': '02d'}],
  'clouds': {'all': 21},
  'wind': {'speed': 2.73, 'deg': 58, 'gust': 5.83},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2026-10-19 15:00:00'},
 {'dt': 1792432800,
  'main': {'temp': 282.12,
   'feels_like': 280.9,
   'temp_min': 280.62,
   'temp_max': 282.12,
   'pressure': 1030,
   'sea_level': 1030,
   'grnd_level': 1025,
   'humidity': 72,
   'temp_kf': 1.5,
   'dew_point': 277.35},
  'weather': [{'id': 802,
    'main': 'Clouds',
    'description': 'scattered clouds',
    'icon': '03n'}],
  'clouds': {'all': 27},
  'wind': {'speed': 2.3, 'deg': 61, 'gust': 4.43},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 

In [14]:
# matches 'cnt' above!
len(response_json['list'])

40

In [15]:
# we have this info in the 'cities' table
response_json['city']

{'id': 6545310,
 'name': 'Mitte',
 'coord': {'lat': 52.52, 'lon': 13.41},
 'country': 'DE',
 'population': 329078,
 'timezone': 7200,
 'sunrise': 1792388300,
 'sunset': 1792425844}

Time to explore one forecast and see what to keep.

In [16]:
forecast = response_json['list'][0]
forecast

{'dt': 1792422000,
 'main': {'temp': 284.75,
  'feels_like': 283.56,
  'temp_min': 284,
  'temp_max': 284.75,
  'pressure': 1031,
  'sea_level': 1031,
  'grnd_level': 1025,
  'humidity': 61,
  'temp_kf': 0.75,
  'dew_point': 277.49},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'clouds': {'all': 21},
 'wind': {'speed': 2.73, 'deg': 58, 'gust': 5.83},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'd'},
 'dt_txt': '2026-10-19 15:00:00'}

* dt looks like some kind of timestamp
* main
    * temp looks crazy! 292 degrees!  (use units=metric instead)
    * feels_like might be more important for people than the actual temperature
    * temp_min/max might be worth keeping? maybe not
    * humidity might affect if people rent scooters or not
* weather looks like the kind of descriptions a weather app would show
* clouds probably not needed
* wind
    * speed or gust could be important
* visibility probably not needed
* pop 'probability of precipitation' seems useful
* rain and snow seem useful
* sys probably not needed
* dt_txt a much better timestamp

In [17]:
forecast['wind']#["feels_like"]

{'speed': 2.73, 'deg': 58, 'gust': 5.83}

In [18]:
forecast_data = {
    "temp": forecast['main']['temp'],
    "feels_like": forecast['main']['feels_like'],
    "humidity_%": forecast['main']['humidity'],
    "wind_speed": forecast['wind']['speed'],
    "wind_gust": forecast['wind']['gust'],
    "precipitation_%": forecast['pop'],
    # # docs say there's 'rain' and 'snow', but it's not here!
    "forecast_time": forecast['dt_txt']
}
forecast_data

{'temp': 284.75,
 'feels_like': 283.56,
 'humidity_%': 61,
 'wind_speed': 2.73,
 'wind_gust': 5.83,
 'precipitation_%': 0,
 'forecast_time': '2026-10-19 15:00:00'}

## One city, all forecasts

Now that we have useful information from one forecast, let's try to extract the same information from the other 39, too.

In [19]:
forecasts = []
for forecast in response_json['list']:
    forecast_data = {
        "temp": forecast['main']['temp'],
        "feels_like": forecast['main']['feels_like'],
        "humidity_%": forecast['main']['humidity'],
        "wind_speed": forecast['wind']['speed'],
        "wind_gust": forecast['wind']['gust'],
        "precipitation_%": forecast['pop'],
        "rain_3h": forecast['rain']['3h'],
        # docs say there's 'snow', but it's not here!
        "forecast_time": forecast['dt_txt']
    }
    forecasts.append(forecast_data)
forecasts

KeyError: 'rain'

Oh no! One of the forecasts has no 'rain'!

In [20]:
forecast # the iteration variable stays stuck wherever the loop broke

{'dt': 1792422000,
 'main': {'temp': 284.75,
  'feels_like': 283.56,
  'temp_min': 284,
  'temp_max': 284.75,
  'pressure': 1031,
  'sea_level': 1031,
  'grnd_level': 1025,
  'humidity': 61,
  'temp_kf': 0.75,
  'dew_point': 277.49},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'clouds': {'all': 21},
 'wind': {'speed': 2.73, 'deg': 58, 'gust': 5.83},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'd'},
 'dt_txt': '2026-10-19 15:00:00'}

Indeed, there's no 'rain' in this dictionary, just like we saw no 'snow' before. JSON documents are semi-structured; where there's no data, they might leave things out rather than listing a null value. Thankfully, python dictionaries have a method that allow us to retrieve a key that might not be there: `.get()`. We can even set the value to return if the key isn't found.

In [21]:
forecast.get('dt_txt')

'2026-10-19 15:00:00'

In [22]:
forecast.get('rain', {}) # {} is the default value

{}

By returning an empty dictionary as the default, we can chain more searches together.

In [23]:
# returns a reasonable value for "no rain in the last three hours"
forecast.get('rain', {}).get('3h', 0)

0

In [24]:
# works just fine if there has been rain
response_json['list'][11].get('rain', {}).get('3h', 0)

0

We'll adjust our loop to use `.get()`

In [25]:
forecasts = []
for forecast in response_json['list']:
    forecast_data = {
        "temp": forecast['main']['temp'],
        "feels_like": forecast['main']['feels_like'],
        "humidity_%": forecast['main']['humidity'],
        "wind_speed": forecast['wind']['speed'],
        "wind_gust": forecast['wind']['gust'],
        "precipitation_%": forecast['pop'],
        "rain_3h": forecast.get('rain', {}).get('3h', 0),
        "snow_3h": forecast.get('snow', {}).get('3h', 0),
        "forecast_time": forecast['dt_txt']
    }
    forecasts.append(forecast_data)
forecasts

[{'temp': 284.75,
  'feels_like': 283.56,
  'humidity_%': 61,
  'wind_speed': 2.73,
  'wind_gust': 5.83,
  'precipitation_%': 0,
  'rain_3h': 0,
  'snow_3h': 0,
  'forecast_time': '2026-10-19 15:00:00'},
 {'temp': 282.12,
  'feels_like': 280.9,
  'humidity_%': 72,
  'wind_speed': 2.3,
  'wind_gust': 4.43,
  'precipitation_%': 0,
  'rain_3h': 0,
  'snow_3h': 0,
  'forecast_time': '2026-10-19 18:00:00'},
 {'temp': 281.52,
  'feels_like': 280.4,
  'humidity_%': 78,
  'wind_speed': 2.05,
  'wind_gust': 4.12,
  'precipitation_%': 0,
  'rain_3h': 0,
  'snow_3h': 0,
  'forecast_time': '2026-10-19 21:00:00'},
 {'temp': 280.99,
  'feels_like': 279.36,
  'humidity_%': 84,
  'wind_speed': 2.58,
  'wind_gust': 5.06,
  'precipitation_%': 0,
  'rain_3h': 0,
  'snow_3h': 0,
  'forecast_time': '2026-10-20 00:00:00'},
 {'temp': 280.06,
  'feels_like': 278.12,
  'humidity_%': 92,
  'wind_speed': 2.75,
  'wind_gust': 5.39,
  'precipitation_%': 0,
  'rain_3h': 0,
  'snow_3h': 0,
  'forecast_time': '2026-1

In [26]:
pd.DataFrame(forecasts)

,temp,feels_like,humidity_%,wind_speed,wind_gust,precipitation_%,rain_3h,snow_3h,forecast_time
0,284.75,283.56,61,2.73,5.83,0.00,0.00,0,2026-10-19 15:00:00
1,282.12,280.90,72,2.30,4.43,0.00,0.00,0,2026-10-19 18:00:00
2,281.52,280.40,78,2.05,4.12,0.00,0.00,0,2026-10-19 21:00:00
3,280.99,279.36,84,2.58,5.06,0.00,0.00,0,2026-10-20 00:00:00
4,280.06,278.12,92,2.75,5.39,0.00,0.00,0,2026-10-20 03:00:00
5,279.37,277.46,94,2.54,5.01,0.00,0.00,0,2026-10-20 06:00:00
6,282.23,280.39,83,3.29,7.70,0.00,0.00,0,2026-10-20 09:00:00
7,283.77,282.79,73,3.32,7.41,0.00,0.00,0,2026-10-20 12:00:00
8,284.12,283.18,73,2.64,5.56,0.00,0.00,0,2026-10-20 15:00:00
9,283.15,281.84,77,2.71,5.52,0.00,0.00,0,2026-10-20 18:00:00


## All cities, all forecasts

Now let's expand one more time by adding a loop to handle each of our three cities.

In [27]:
cities_df

,city_id,name,country,latitude,longitude
0,1,Berlin,Germany,52.52,13.41
1,2,Hamburg,Germany,53.55,10.00
2,3,Munich,Germany,48.14,11.58


In [28]:
forecasts = []
url = "https://api.openweathermap.org/data/2.5/forecast"

# loop through cities_dfa
for _, row in cities_df.iterrows():
    # don't forget to add a parameter to use metric units!
    params = {"lat": row["latitude"], "lon" :row["longitude"], "appid":weather_key, "units": "metric"}
    response = requests.get(url, params=params)
    response_json = response.json()
    
    for forecast in response_json['list']:
        forecast_data = {
            "temp": forecast['main']['temp'],
            "feels_like": forecast['main']['feels_like'],
            "humidity_%": forecast['main']['humidity'],
            "wind_speed": forecast['wind']['speed'],
            "wind_gust": forecast['wind']['gust'],
            "precipitation_%": forecast['pop'],
            "rain_3h": forecast.get('rain', {}).get('3h', 0),
            "snow_3h": forecast.get('snow', {}).get('3h', 0),
            "forecast_time": forecast['dt_txt'],
            "city_id": row["city_id"] # add city_id to reference back to 'cities' table
        }
        forecasts.append(forecast_data)

forecast_df = pd.DataFrame(forecasts)
forecast_df

,temp,feels_like,humidity_%,wind_speed,wind_gust,precipitation_%,rain_3h,snow_3h,forecast_time,city_id
0,11.60,10.41,61,2.73,5.83,0.00,0.00,0,2026-10-19 15:00:00,1
1,8.97,7.75,72,2.30,4.43,0.00,0.00,0,2026-10-19 18:00:00,1
2,8.37,7.25,78,2.05,4.12,0.00,0.00,0,2026-10-19 21:00:00,1
3,7.84,6.21,84,2.58,5.06,0.00,0.00,0,2026-10-20 00:00:00,1
4,6.91,4.97,92,2.75,5.39,0.00,0.00,0,2026-10-20 03:00:00,1
...,...,...,...,...,...,...,...,...,...,...
115,6.79,3.28,80,5.72,12.03,0.00,0.00,0,2026-10-24 00:00:00,3
116,6.06,2.63,77,5.07,10.38,0.00,0.00,0,2026-10-24 03:00:00,3
117,5.53,2.43,74,4.15,8.27,0.00,0.00,0,2026-10-24 06:00:00,3
118,9.54,7.54,56,3.79,8.74,0.00,0.00,0,2026-10-24 09:00:00,3


In [29]:
# useful for writing the SQL table definition
forecast_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   temp             120 non-null    float64
 1   feels_like       120 non-null    float64
 2   humidity_%       120 non-null    int64  
 3   wind_speed       120 non-null    float64
 4   wind_gust        120 non-null    float64
 5   precipitation_%  120 non-null    float64
 6   rain_3h          120 non-null    float64
 7   snow_3h          120 non-null    int64  
 8   forecast_time    120 non-null    str    
 9   city_id          120 non-null    int64  
dtypes: float64(6), int64(3), str(1)
memory usage: 9.5 KB


In [30]:
# forecast_df.to_sql(
#     "forecasts",
#     con=connection_string,
#     if_exists="append",
#     index=False
# )